# BERT

In this lab, we will use a pre-trained BERT model to make embeddings of a dataset of synthesis recipes of materials nanoparticles.

In [1]:
try:
    import google.colab
    IN_COLAB = True
    !git clone https://github.com/dskoda/ml4mat-26s-public.git
    !cd ml4mat-26s-public && pip install . && cd ..
    !pip install torch torchvision
    !pip install tqdm
    ROOT = "https://raw.githubusercontent.com/dskoda/ml4mat-26s-public/refs/heads/main/lectures/12-RNNs"
    STYLE = "colab"
except:
    IN_COLAB = False
    ROOT = "."
    STYLE = "jupyter"

In [2]:
import tqdm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import torch
from transformers import BertTokenizer, BertModel, AutoTokenizer, AutoModelForMaskedLM, pipeline

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn import metrics

## Loading the synthesis data

In [3]:
df = pd.read_csv(f"{ROOT}/data/aunp-data-classification.csv")
df = df.loc[(df["type"] == "materials") | (df["type"] == "synthesis")].dropna()
df.head()

,DOI,type,text
0,10.1002/adfm.200902372,materials,"Toluene (>99%), methanol (99.8%), and acetone ..."
1,10.1002/adfm.200902372,synthesis,Solutions of 1 M trioctylphosphine selenide (T...
2,10.1002/adfm.200902372,synthesis,NW Synthesis: CdSe NWs were synthesized follow...
3,10.1002/adfm.200902372,synthesis,QD Synthesis: Colloidal CdSe QDs were synthesi...
4,10.1002/adfm.200902372,synthesis,"In parallel, an injection solution of 1 M TOPS..."


## Downloading the pre-trained BERT model

In [4]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
    
batch_size = 100

In [5]:
model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)

model.to(device)
model.eval()  # Set model to evaluation mode

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

## Evaluating the model

Now, we will tokenize and evaluate the model.
With the produced embeddings, we will do some operation on the latent space of materials synthesis recipes.

In [6]:
embeddings = []

with torch.no_grad():
    # Iterate over the sentences in batches
    for i in tqdm.tqdm(range(0, len(df), batch_size)):
        batch = df.iloc[i:i + batch_size]["text"].values.tolist()
        
        # Tokenize the batch with padding and truncation
        encoded_input = tokenizer(batch, padding=True, truncation=True, return_tensors='pt')
        
        # Move the inputs to the specified device
        encoded_input = {key: tensor.to(device) for key, tensor in encoded_input.items()}
        
        outputs = model(**encoded_input)
        
        # Extract the embeddings from the [CLS] token (pooler_output)
        batch_embeddings = outputs.pooler_output  # Shape: (batch_size, hidden_size)
        embeddings.append(batch_embeddings.cpu().numpy())
        
X = np.concatenate(embeddings, axis=0)
print("Embeddings shape:", X.shape)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 30/30 [01:05<00:00,  2.18s/it]

Embeddings shape: (2928, 768)


### With the embeddings, let's train a simple RF model:

In [7]:
y = (df["type"].values == "synthesis").astype(int)

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.3)

In [9]:
rf = RandomForestClassifier()
rf.fit(X_train, y_train)

RandomForestClassifier()

In [10]:
y_pred = rf.predict(X_test)
print(metrics.classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.72      0.58      0.64       343
           1       0.76      0.85      0.80       536

    accuracy                           0.75       879
   macro avg       0.74      0.72      0.72       879
weighted avg       0.74      0.75      0.74       879



### Exercise

Now that you have seen how reasonably easy it is to load a pre-trained model, why not try with other models?
Explore the library of pre-trained models on [HuggingFace](https://huggingface.co/models) and see if you can run any of them.